# Laboratorio 1.2 — Entrena tu primer modelo de Machine Learning

Módulo 1 · Introducción a la Inteligencia Artificial — bloque [`02-machine-learning-y-deep-learning.md`](../../../Apuntes-Markdown/01-introduccion-a-la-ia/02-machine-learning-y-deep-learning.md)

En este notebook vas a entrenar un modelo de **aprendizaje supervisado** que predice si un tumor es **maligno** o **benigno** a partir de un conjunto de medidas del núcleo celular obtenidas de una biopsia. El dataset es real, público y viene ya incluido en `scikit-learn` — no necesitas descargar ningún fichero.

**Cómo usar este notebook**: todo el código ya está escrito y funciona ejecutando las celdas en orden, de arriba a abajo, sin tocar nada. Cuando el enunciado del laboratorio te lo indique, volverás a algunas celdas concretas (marcadas con ✏️) para cambiar un valor y volver a ejecutar desde ahí hacia abajo.

No hace falta saber programar en profundidad: lo importante es que entiendas **qué** hace cada celda y **qué relación** tiene con los conceptos del apunte (features, target, entrenamiento, validación e inferencia).

## Paso 0 — Importar las librerías

Ejecuta esta celda primero. Carga las librerías que se usan en el resto del notebook: `pandas` para manejar los datos en forma de tabla, `sklearn` (scikit-learn) para el dataset y los modelos de Machine Learning, y `matplotlib` para las gráficas.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

# Semilla fija para que el ejercicio sea reproducible entre distintos alumnos
SEMILLA = 42

print("Librerías cargadas correctamente.")

## Paso 1 — Cargar el dataset

Cargamos el dataset "breast cancer" directamente desde `scikit-learn` con `load_breast_cancer(as_frame=True)`, que nos lo entrega ya como una tabla de `pandas` (DataFrame). Cada fila es una biopsia; cada columna (salvo la última) es una medida numérica del núcleo celular; la columna `target` es la etiqueta que queremos predecir:

- `target = 0` → tumor **maligno**
- `target = 1` → tumor **benigno**

Esta es exactamente la estructura de "Datos, features y etiquetas" del apunte: cada fila es una observación, cada columna de entrada es una feature, y la última columna es el target.

In [ ]:
datos = load_breast_cancer(as_frame=True)
df = datos.frame

print(f"Número de observaciones (filas): {df.shape[0]}")
print(f"Número de columnas totales (features + target): {df.shape[1]}")
print()
print("Significado de la etiqueta target:")
for valor, nombre in enumerate(datos.target_names):
    print(f"  target = {valor}  ->  {nombre}")
print()
print("Distribución de la variable objetivo:")
print(df["target"].value_counts().rename(index={0: "maligno (0)", 1: "benigno (1)"}))

df.head()

## Paso 2 — Elegir las features (variables de entrada)

El dataset completo tiene 30 columnas de medidas distintas, lo cual es difícil de interpretar para un primer contacto con ML. Para este laboratorio usamos un subconjunto de **8 columnas legibles**, todas ellas medidas "promedio" (`mean`) de propiedades del núcleo celular:

- `mean radius`: radio medio del núcleo.
- `mean texture`: variación de intensidad en la escala de grises.
- `mean perimeter`: perímetro medio del núcleo.
- `mean area`: área media del núcleo.
- `mean smoothness`: variación local de longitud de radio (suavidad del contorno).
- `mean concavity`: severidad de las partes cóncavas del contorno.
- `mean symmetry`: simetría del núcleo.
- `mean concave points`: número de partes cóncavas del contorno.

✏️ **Puedes cambiar esta lista** si quieres experimentar con otras columnas (por ejemplo, añadiendo alguna que empiece por `worst`), pero para la actividad guiada del laboratorio no es necesario tocar nada aquí.

In [ ]:
FEATURES = [
    "mean radius",
    "mean texture",
    "mean perimeter",
    "mean area",
    "mean smoothness",
    "mean concavity",
    "mean symmetry",
    "mean concave points",
]

X = df[FEATURES]
y = df["target"]

print(f"X (features) tiene forma: {X.shape}")
print(f"y (target) tiene forma: {y.shape}")

X.describe()

## Paso 3 — Partición en entrenamiento y validación (train/test split)

Antes de entrenar, separamos los datos en dos conjuntos (según el apunte):

- **Entrenamiento (train)**: los ejemplos que el modelo usa para ajustar sus parámetros.
- **Validación (test)**: ejemplos que el modelo NO ve durante el entrenamiento, y que usamos después para comprobar si generaliza bien a casos nuevos.

La variable `TEST_SIZE` controla qué proporción de los datos se reserva para validación. Por ejemplo, `TEST_SIZE = 0.2` significa que el 20% de los datos se usa para validar y el 80% restante para entrenar.

✏️ **Esta es una de las dos variables que vas a modificar durante la Fase 4 del laboratorio.** Por ahora, déjala en su valor por defecto (`0.2`) y ejecuta la celda.

In [ ]:
TEST_SIZE = 0.2  # ✏️ cámbialo a 0.1 o a 0.5 en la Fase 4 del laboratorio y vuelve a ejecutar desde aquí

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEMILLA, stratify=y
)

print(f"TEST_SIZE actual: {TEST_SIZE}")
print(f"Ejemplos de entrenamiento: {X_train.shape[0]}")
print(f"Ejemplos de validación:    {X_test.shape[0]}")

## Paso 4 — Elegir el algoritmo

Vamos a comparar dos algoritmos de clasificación distintos, ambos vistos como ejemplos de aprendizaje supervisado en el apunte:

- `"arbol"` → un **árbol de decisión** (`DecisionTreeClassifier`): aprende una secuencia de reglas de tipo "si `mean radius` > X entonces...".
- `"logistica"` → una **regresión logística** (`LogisticRegression`): aprende una combinación ponderada de las features para estimar la probabilidad de cada clase.

La variable `MODELO` decide cuál de los dos se instancia y se entrena.

✏️ **Esta es la segunda variable que vas a modificar durante la Fase 4 del laboratorio.** Por ahora, déjala en `"arbol"` y ejecuta la celda.

In [ ]:
MODELO = "arbol"  # ✏️ cámbialo a "logistica" en la Fase 4 del laboratorio y vuelve a ejecutar desde aquí

if MODELO == "arbol":
    modelo = DecisionTreeClassifier(max_depth=4, random_state=SEMILLA)
elif MODELO == "logistica":
    modelo = LogisticRegression(max_iter=5000, random_state=SEMILLA)
else:
    raise ValueError('MODELO debe ser "arbol" o "logistica"')

print(f"Algoritmo seleccionado: {MODELO}")
print(modelo)

## Paso 5 — Entrenamiento

Con `.fit(X_train, y_train)` el modelo ajusta sus parámetros internos usando únicamente el conjunto de entrenamiento. Esta es la fase de "Entrenamiento" del apunte: el algoritmo minimiza el error sobre los datos de entrenamiento (concepto de "Función de pérdida y optimización").

In [ ]:
modelo.fit(X_train, y_train)
print("Modelo entrenado.")

## Paso 6 — Inferencia y evaluación

Ahora usamos el modelo ya entrenado para predecir sobre el conjunto de validación (`X_test`), que el modelo no ha visto durante el entrenamiento — esta es la fase de "Inferencia" del apunte, aplicada aquí con fines de validación.

La función `imprimir_metricas` ya está escrita y calcula tres métricas estándar de clasificación:

- **Accuracy**: proporción de predicciones correctas sobre el total.
- **Precision**: de los casos que el modelo predijo como malignos, ¿qué proporción lo eran realmente?
- **Recall**: de los casos que realmente eran malignos, ¿qué proporción detectó el modelo? (En diagnóstico médico, un recall bajo significa muchos falsos negativos — tumores malignos no detectados —, que suele ser el error más costoso de evitar).

In [ ]:
def imprimir_metricas(y_real, y_pred, titulo=""):
    """Imprime accuracy, precision y recall de forma legible.
    pos_label=0 porque la clase 'maligno' (0) es la clase positiva de interés clínico."""
    acc = accuracy_score(y_real, y_pred)
    prec = precision_score(y_real, y_pred, pos_label=0)
    rec = recall_score(y_real, y_pred, pos_label=0)

    if titulo:
        print(f"--- {titulo} ---")
    print(f"Accuracy:  {acc:.3f}  (proporción total de aciertos)")
    print(f"Precision (clase 'maligno'): {prec:.3f}  (de lo predicho como maligno, cuánto lo era)")
    print(f"Recall (clase 'maligno'):    {rec:.3f}  (de los malignos reales, cuántos detectó)")
    return acc, prec, rec


y_pred = modelo.predict(X_test)

print(f"Configuración actual -> TEST_SIZE={TEST_SIZE}, MODELO='{MODELO}'\n")
_ = imprimir_metricas(y_test, y_pred, titulo="Resultado en el conjunto de validación")

## Paso 7 — Matriz de confusión

La matriz de confusión muestra, de un vistazo, los aciertos y los dos tipos de error posibles: falsos positivos (benignos clasificados como malignos) y falsos negativos (malignos clasificados como benignos, la casilla superior derecha si `maligno` está en la primera fila).

In [ ]:
matriz = confusion_matrix(y_test, y_pred, labels=[0, 1])

disp = ConfusionMatrixDisplay(
    confusion_matrix=matriz,
    display_labels=["maligno (0)", "benigno (1)"],
)
fig, ax = plt.subplots(figsize=(5, 5))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Matriz de confusión — TEST_SIZE={TEST_SIZE}, MODELO='{MODELO}'")
plt.tight_layout()
plt.show()

## Paso 8 — Experimentación guiada (Fase 4 del laboratorio)

Ahora te toca experimentar. Sigue estos pasos:

1. Sube a la celda del **Paso 3** y cambia `TEST_SIZE` a `0.1`. Vuelve a ejecutar, en orden, las celdas del Paso 3 en adelante (Paso 3 → 4 → 5 → 6 → 7). Anota accuracy, precision y recall.
2. Cambia `TEST_SIZE` a `0.5` y repite.
3. Vuelve a dejar `TEST_SIZE = 0.2`, y esta vez cambia `MODELO` (celda del **Paso 4**) a `"logistica"`. Vuelve a ejecutar desde el Paso 4 en adelante y anota los resultados.
4. Si te queda tiempo, combina cambios: prueba `TEST_SIZE = 0.1` con `MODELO = "logistica"`, por ejemplo.

La siguiente celda de código es una utilidad opcional para que puedas comparar varias configuraciones **en una sola tabla**, sin tener que ir anotando a mano en un papel. No es obligatoria para completar el laboratorio, pero te puede ayudar a comparar de un vistazo.

In [ ]:
def experimento(test_size, nombre_modelo):
    """Repite el ciclo completo (split -> entrenar -> evaluar) para una combinación
    de TEST_SIZE y MODELO, y devuelve un diccionario con las métricas obtenidas.
    Útil para comparar varias configuraciones sin tener que ejecutar celdas a mano una por una."""
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=test_size, random_state=SEMILLA, stratify=y
    )

    if nombre_modelo == "arbol":
        m = DecisionTreeClassifier(max_depth=4, random_state=SEMILLA)
    elif nombre_modelo == "logistica":
        m = LogisticRegression(max_iter=5000, random_state=SEMILLA)
    else:
        raise ValueError('nombre_modelo debe ser "arbol" o "logistica"')

    m.fit(X_tr, y_tr)
    y_pred_local = m.predict(X_te)

    return {
        "TEST_SIZE": test_size,
        "MODELO": nombre_modelo,
        "n_train": len(X_tr),
        "n_test": len(X_te),
        "accuracy": round(accuracy_score(y_te, y_pred_local), 3),
        "precision_maligno": round(precision_score(y_te, y_pred_local, pos_label=0), 3),
        "recall_maligno": round(recall_score(y_te, y_pred_local, pos_label=0), 3),
    }


configuraciones = [
    (0.1, "arbol"),
    (0.2, "arbol"),
    (0.5, "arbol"),
    (0.1, "logistica"),
    (0.2, "logistica"),
    (0.5, "logistica"),
]

resultados = pd.DataFrame([experimento(ts, m) for ts, m in configuraciones])
resultados

**Cómo leer la tabla anterior**: fíjate especialmente en la columna `n_train` (número de ejemplos de entrenamiento) para las filas con `TEST_SIZE = 0.5`. Cuantos menos ejemplos ve el modelo durante el entrenamiento, más difícil le resulta aprender un patrón fiable — y cuantos menos ejemplos quedan para validación (`TEST_SIZE = 0.1`), menos fiable es la métrica de validación en sí misma, porque se calcula sobre muy pocos casos. Ambos extremos tienen un coste distinto: es el mismo equilibrio del que habla el apunte al distinguir entrenamiento y validación.

## Tu conclusión (escribe aquí)

Responde en esta celda, con tus propias palabras, en 3-4 líneas:

1. ¿Cómo cambiaron accuracy, precision y recall al mover `TEST_SIZE` entre 0.1 y 0.5? ¿Qué configuración te pareció más fiable y por qué?
2. ¿Hubo diferencias notables entre el árbol de decisión y la regresión logística en este dataset?
3. Conecta lo que observaste con la idea de overfitting/underfitting: ¿en qué configuración crees que el modelo estaba aprendiendo mejor a generalizar, y en cuál corría más riesgo de sobreajustarse o de no tener suficientes datos para aprender bien?

_(Escribe tu respuesta aquí, sustituyendo este texto)_
